# Session 2, Module 08: Magic Methods and Dunder


This module covers:
- __repr__ and __str__ — debugging vs display
- __len__, __getitem__, __setitem__ — container behavior
- __eq__, __lt__, __hash__ — comparison and hashing
- __enter__ and __exit__ — context manager protocol
- __call__ — making objects callable
- __iter__ and __next__ — making objects iterable

Data Engineering Context:
Magic methods let you build custom data structures that behave like
built-in types, making your code more Pythonic and intuitive.


In [ ]:
from typing import Iterator, Any

============================================================
__repr__ AND __str__ — Object Representation
============================================================

In [ ]:
print("=== __repr__ and __str__ ===")


class Record:
    """Demonstrates __repr__ and __str__."""

    def __init__(self, id: int, name: str, value: float):
        self.id = id
        self.name = name
        self.value = value

    def __repr__(self) -> str:
        """
        Developer representation — unambiguous, ideally eval-able.
        Used in debugger, REPL, and when __str__ is not defined.
        """
        return f"Record(id={self.id!r}, name={self.name!r}, value={self.value!r})"

    def __str__(self) -> str:
        """
        User-friendly representation — readable, for display.
        Used by print() and str().
        """
        return f"Record #{self.id}: {self.name} (${self.value:.2f})"


record = Record(1, "Widget", 19.99)

# __repr__ — for developers
print(f"repr(record): {repr(record)}")
# OUTPUT: Record(id=1, name='Widget', value=19.99)

# __str__ — for users
print(f"str(record): {str(record)}")
# OUTPUT: Record #1: Widget ($19.99)

# print() uses __str__
print(f"print(record): {record}")

# In containers, __repr__ is used
records = [Record(1, "A", 10), Record(2, "B", 20)]
print(f"List of records: {records}")

============================================================
__len__, __getitem__, __setitem__ — Container Behavior
============================================================

In [ ]:
print("\n=== Container Behavior ===")


class DataBatch:
    """
    A batch of records that behaves like a container.

    Implements len(), indexing, and slicing.
    """

    def __init__(self, records: list[dict] = None):
        self._records = records or []

    def __len__(self) -> int:
        """Return number of records. Enables len(batch)."""
        return len(self._records)

    def __getitem__(self, index: int | slice) -> dict | list[dict]:
        """
        Get record(s) by index. Enables batch[0], batch[1:3].
        """
        return self._records[index]

    def __setitem__(self, index: int, value: dict) -> None:
        """Set record at index. Enables batch[0] = {...}."""
        self._records[index] = value

    def __delitem__(self, index: int) -> None:
        """Delete record at index. Enables del batch[0]."""
        del self._records[index]

    def __contains__(self, item: dict) -> bool:
        """Check if record exists. Enables 'x in batch'."""
        return item in self._records

    def append(self, record: dict) -> None:
        """Add record to batch."""
        self._records.append(record)

    def __repr__(self) -> str:
        return f"DataBatch({len(self)} records)"


# Create batch
batch = DataBatch([
    {"id": 1, "name": "Alice"},
    {"id": 2, "name": "Bob"},
    {"id": 3, "name": "Charlie"},
])

# len() works
print(f"len(batch): {len(batch)}")

# Indexing works
print(f"batch[0]: {batch[0]}")
print(f"batch[-1]: {batch[-1]}")

# Slicing works
print(f"batch[1:3]: {batch[1:3]}")

# Assignment works
batch[0] = {"id": 1, "name": "Alice Smith"}
print(f"After assignment: {batch[0]}")

# Containment check works
test_record = {"id": 2, "name": "Bob"}
print(f"{test_record} in batch: {test_record in batch}")

============================================================
__eq__, __lt__, __hash__ — Comparison and Hashing
============================================================

In [ ]:
print("\n=== Comparison and Hashing ===")


class Version:
    """
    Semantic version that supports comparison.

    Implements ==, <, <=, >, >= through __eq__ and __lt__.
    """

    def __init__(self, major: int, minor: int, patch: int):
        self.major = major
        self.minor = minor
        self.patch = patch

    def __eq__(self, other: "Version") -> bool:
        """Equality comparison. Enables v1 == v2."""
        if not isinstance(other, Version):
            return NotImplemented
        return (self.major, self.minor, self.patch) == (other.major, other.minor, other.patch)

    def __lt__(self, other: "Version") -> bool:
        """Less than comparison. Enables v1 < v2."""
        if not isinstance(other, Version):
            return NotImplemented
        return (self.major, self.minor, self.patch) < (other.major, other.minor, other.patch)

    def __le__(self, other: "Version") -> bool:
        """Less than or equal. Enables v1 <= v2."""
        return self == other or self < other

    def __gt__(self, other: "Version") -> bool:
        """Greater than. Enables v1 > v2."""
        return not self <= other

    def __ge__(self, other: "Version") -> bool:
        """Greater than or equal. Enables v1 >= v2."""
        return not self < other

    def __hash__(self) -> int:
        """
        Hash for use in sets/dicts. Required if __eq__ is defined.
        Objects that compare equal must have the same hash.
        """
        return hash((self.major, self.minor, self.patch))

    def __repr__(self) -> str:
        return f"Version({self.major}, {self.minor}, {self.patch})"

    def __str__(self) -> str:
        return f"{self.major}.{self.minor}.{self.patch}"


v1 = Version(1, 0, 0)
v2 = Version(1, 2, 0)
v3 = Version(1, 2, 0)
v4 = Version(2, 0, 0)

print(f"v1 = {v1}, v2 = {v2}, v3 = {v3}, v4 = {v4}")
print(f"v1 == v2: {v1 == v2}")
print(f"v2 == v3: {v2 == v3}")
print(f"v1 < v2: {v1 < v2}")
print(f"v4 > v2: {v4 > v2}")

# Can sort versions
versions = [v4, v1, v2]
print(f"\nSorted: {sorted(versions)}")

# Can use as dict key (because __hash__ is defined)
version_features = {v1: ["basic"], v2: ["advanced"], v4: ["enterprise"]}
print(f"Features for {v2}: {version_features[v2]}")

# Can use in set
unique_versions = {v1, v2, v3}  # v2 and v3 deduplicate
print(f"Unique versions: {unique_versions}")

============================================================
__enter__ AND __exit__ — Context Manager Protocol
============================================================

In [ ]:
print("\n=== Context Manager Protocol ===")


class DatabaseConnection:
    """
    Database connection that works with 'with' statement.

    Ensures proper cleanup even if exceptions occur.
    """

    def __init__(self, connection_string: str):
        self.connection_string = connection_string
        self.connection = None

    def __enter__(self) -> "DatabaseConnection":
        """
        Called when entering 'with' block.
        Returns the context object (usually self).
        """
        print(f"Connecting to {self.connection_string}...")
        self.connection = {"active": True}  # Simulated connection
        return self

    def __exit__(self, exc_type, exc_val, exc_tb) -> bool:
        """
        Called when exiting 'with' block (always, even on exception).

        Args:
            exc_type: Exception type if error occurred, else None
            exc_val: Exception value if error occurred, else None
            exc_tb: Traceback if error occurred, else None

        Returns:
            True to suppress exception, False to propagate
        """
        print(f"Closing connection...")
        self.connection = None

        if exc_type is not None:
            print(f"Exception occurred: {exc_type.__name__}: {exc_val}")
            # Return False to propagate exception
            return False

        return True

    def execute(self, query: str) -> list:
        """Execute a query."""
        if not self.connection:
            raise RuntimeError("Not connected")
        print(f"Executing: {query}")
        return [{"result": "data"}]


# Using with statement
print("Normal execution:")
with DatabaseConnection("postgresql://localhost/db") as db:
    results = db.execute("SELECT * FROM users")
    print(f"Results: {results}")
# Connection automatically closed!

print("\nWith exception:")
try:
    with DatabaseConnection("postgresql://localhost/db") as db:
        raise ValueError("Something went wrong!")
except ValueError:
    print("Exception was propagated")

============================================================
__call__ — Callable Objects
============================================================

In [ ]:
print("\n=== __call__ — Callable Objects ===")


class Transformer:
    """
    A callable transformer that can be used like a function.

    Useful for configurable transformations.
    """

    def __init__(self, prefix: str = "", suffix: str = ""):
        self.prefix = prefix
        self.suffix = suffix
        self.call_count = 0

    def __call__(self, value: str) -> str:
        """
        Called when instance is invoked as function.
        Enables: transformer("value")
        """
        self.call_count += 1
        return f"{self.prefix}{value}{self.suffix}"


# Create transformer instance
add_quotes = Transformer(prefix='"', suffix='"')

# Use it like a function!
result = add_quotes("hello")
print(f'add_quotes("hello"): {result}')

# Apply to multiple values
values = ["one", "two", "three"]
quoted = [add_quotes(v) for v in values]
print(f"Quoted values: {quoted}")
print(f"Call count: {add_quotes.call_count}")

# Use with map()
snake_case = Transformer(suffix="_col")
columns = list(map(snake_case, ["id", "name", "email"]))
print(f"Transformed columns: {columns}")

============================================================
__iter__ AND __next__ — Iteration Protocol
============================================================

In [ ]:
print("\n=== __iter__ and __next__ — Iteration ===")


class RecordIterator:
    """
    Custom iterator for records.

    Demonstrates the iteration protocol.
    """

    def __init__(self, records: list[dict]):
        self._records = records
        self._index = 0

    def __iter__(self) -> "RecordIterator":
        """Return the iterator object (self)."""
        return self

    def __next__(self) -> dict:
        """Return next item or raise StopIteration."""
        if self._index >= len(self._records):
            raise StopIteration

        record = self._records[self._index]
        self._index += 1
        return record


class DataSet:
    """
    A dataset that is iterable.

    Returns a new iterator each time, so it can be iterated multiple times.
    """

    def __init__(self, records: list[dict]):
        self._records = records

    def __iter__(self) -> Iterator[dict]:
        """Return a new iterator. Enables 'for record in dataset'."""
        return RecordIterator(self._records)

    def __len__(self) -> int:
        return len(self._records)


# Create dataset
dataset = DataSet([
    {"id": 1, "value": 100},
    {"id": 2, "value": 200},
    {"id": 3, "value": 300},
])

# Iterate with for loop
print("First iteration:")
for record in dataset:
    print(f"  {record}")

# Can iterate again (new iterator each time)
print("\nSecond iteration:")
for record in dataset:
    print(f"  {record}")

# Works with list(), sum(), etc.
all_records = list(dataset)
total = sum(r["value"] for r in dataset)
print(f"\nTotal records: {len(all_records)}, Total value: {total}")

## Practical: Custom Data Batch


In [ ]:
print("\n=== Practical: Custom DataBatch ===")


class DataBatchAdvanced:
    """
    A feature-rich data batch implementing multiple protocols.
    """

    def __init__(self, records: list[dict] = None, name: str = "batch"):
        self._records = records or []
        self.name = name

    # Representation
    def __repr__(self) -> str:
        return f"DataBatch(name={self.name!r}, records={len(self)})"

    def __str__(self) -> str:
        return f"{self.name}: {len(self)} records"

    # Container
    def __len__(self) -> int:
        return len(self._records)

    def __getitem__(self, index):
        return self._records[index]

    def __setitem__(self, index, value):
        self._records[index] = value

    def __contains__(self, item):
        return item in self._records

    # Iteration
    def __iter__(self):
        return iter(self._records)

    # Comparison
    def __eq__(self, other):
        if not isinstance(other, DataBatchAdvanced):
            return NotImplemented
        return self._records == other._records

    # Callable — filter records
    def __call__(self, predicate) -> "DataBatchAdvanced":
        """Filter records using predicate function."""
        filtered = [r for r in self._records if predicate(r)]
        return DataBatchAdvanced(filtered, f"{self.name}_filtered")

    # Addition — combine batches
    def __add__(self, other: "DataBatchAdvanced") -> "DataBatchAdvanced":
        """Combine two batches."""
        return DataBatchAdvanced(
            self._records + other._records,
            f"{self.name}+{other.name}"
        )


# Create and use batch
batch = DataBatchAdvanced([
    {"id": 1, "status": "active", "value": 100},
    {"id": 2, "status": "inactive", "value": 50},
    {"id": 3, "status": "active", "value": 200},
], name="sales")

print(f"Batch: {batch}")
print(f"Length: {len(batch)}")
print(f"First: {batch[0]}")

# Filter using __call__
active = batch(lambda r: r["status"] == "active")
print(f"Filtered: {active}")

# Iterate
print("Records:")
for record in batch:
    print(f"  {record}")

# Combine using __add__
batch2 = DataBatchAdvanced([{"id": 4, "status": "active", "value": 75}], "new")
combined = batch + batch2
print(f"\nCombined: {combined}")

## Summary


In [ ]:
print("\n=== Summary ===")
print("""
Representation:
  __repr__: Developer representation (debugging)
  __str__:  User-friendly representation (display)

Container:
  __len__:      len(obj)
  __getitem__:  obj[key]
  __setitem__:  obj[key] = value
  __delitem__:  del obj[key]
  __contains__: item in obj

Comparison:
  __eq__: ==    __ne__: !=
  __lt__: <     __le__: <=
  __gt__: >     __ge__: >=
  __hash__: Required for dict keys/set elements

Context Manager:
  __enter__: Called on 'with' entry
  __exit__:  Called on 'with' exit (even on exception)

Callable:
  __call__: obj() — makes instance callable like function

Iteration:
  __iter__: Returns iterator
  __next__: Returns next item or raises StopIteration

Best Practices:
  - Always define __repr__ for debugging
  - Define __str__ if user-friendly display needed
  - If __eq__ defined, define __hash__ (or set to None)
  - Return NotImplemented for unsupported comparisons
""")